# Autoencoder: reconstructing MNIST

Compress digits into a small latent vector, then reconstruct pixels. Labels are not used by the reconstruction objective.

This notebook starts in **offline smoke mode**. It verifies wiring on tiny synthetic batches, not dataset performance. Switch `SMOKE` to `False` for the named experiment.

## Setup

Install this checkout with `python -m pip install -e ".[notebooks]"` and select that environment as the notebook kernel. The following cell supports launching from the repository root or notebooks folder.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path.cwd().parent
if not (ROOT / 'examples').exists():
    raise RuntimeError('Launch Jupyter from the repository root or notebooks directory')
sys.path.insert(0, str(ROOT))
import neuraljaxwork as nj
from neuraljaxwork import models, inspection
from examples.run import run_experiment
from IPython.display import display, Image
import json

## Architecture and tensor contract

Input/output `[B,1,28,28]`; latent `[B,8]`. The decoder sigmoid constrains reconstruction to `[0,1]`; MSE averages over all pixels.

The summary is a module hierarchy with deduplicated parameters, not a complete computation diagram.

In [ ]:
model = models.Autoencoder(latent_size=8, rngs=nj.Rngs(params=0))
print(inspection.summary(model))

## Experiment controls

`SMOKE=True` never downloads datasets. For a real image/Titanic run, set `SMOKE=False` and explicitly allow the first download. Increase the training budget gradually; no full training is part of the default test suite.

In [ ]:
SMOKE = True
DOWNLOAD = False
EPOCHS = 1 if SMOKE else 10
LIMIT = 512
SEED = 0
# For transformer experiments, optionally set a permitted ASCII corpus path.
TEXT_FILE = None

## Train, validate, and save

The shared runner uses explicit update functions in `neuraljaxwork.training`. Supervised examples select the best validation checkpoint before final test evaluation. GANs save both participants and fixed-noise galleries. Every run uses a fresh directory.

In [ ]:
output = run_experiment(
    'autoencoder', smoke=SMOKE, download=DOWNLOAD, epochs=EPOCHS,
    limit=LIMIT, seed=SEED, cache=str(ROOT / 'data'), text_file=TEXT_FILE,
    output=str(ROOT / 'runs' / ('autoencoder' + '-notebook-' + str(__import__('time').time_ns()))),
)
print(output)

## Inspect evidence

Smoke results are labeled in the report. Treat them only as a wiring check. Full runs should be compared against the documented baseline and budget.

In [ ]:
report = json.loads((output / 'report.json').read_text())
display(report)
display(Image(filename=str(output / 'learning.png')))

In [ ]:
artifact = output / 'reconstructions.png'
if artifact.exists():
    display(Image(filename=str(artifact)))
if (output / 'generation.txt').exists():
    print((output / 'generation.txt').read_text())

## Interpretation and next experiment

Compare adjacent original/reconstruction pairs. A model may blur digits while achieving low MSE. Saved latent codes permit a later PCA plot; projections are descriptive and do not prove semantic disentanglement.

Change one variable at a time, use fresh run paths, and keep the seed/configuration. See `docs/examples.md` for provenance, caveats, and longer commands. No quality scores are prefilled into this notebook.